In [1]:
from sympy import dsolve, limit, simplify, diff, solve, integrate, summation
from sympy import symbols, sqrt  
from sympy import Function, Eq, coth, exp, cos, sin, asin, cosh, sinh, IndexedBase
from sympy import oo, latex

from sympy.vector import CoordSys3D
from sympy.vector import cross, dot

import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import interact, FloatSlider

In [2]:
N = CoordSys3D('N')

i, j, k = N.i, N.j, N.k

<font color='violet'> ***Приложение к лекциям по аналитической механике (Задача 21.13)*** <font>

Рассмотрим одномерный гармонический осциллятор с Лагранжианом:

$$L = \frac{1}{2} (\dot q^2 - \omega^2 q^2) $$

Найдём функцию q(t), решив задачу Коши

$$ \frac{d}{dt} \frac{L}{\partial \dot q} - \frac{\partial L}{\partial q} = 0, \; q(0) = q_0, \; \dot q(0) = 0$$

In [3]:
t = symbols('t')
q = Function('q')(t)
omega, q_0 = symbols('omega q_0', real=True, positive=True)

L = (q.diff(t)**2 - omega**2 * q**2) / 2
lagrange_equation = Eq((L.diff(q.diff(t))).diff(t) - L.diff(q), 0)
lagrange_equation

initial_condition = {q.subs(t, 0): q_0, q.diff(t).subs(t, 0): 0}
solution = dsolve(lagrange_equation, q, ics=initial_condition)

ans = simplify(solution).rhs

q_straight_eq = Eq(q, ans)
q_straight_eq

Eq(q(t), q_0*cos(omega*t))

Рассчитаем действие Гамильтона по прямому пути 
$$ q_{\text{пр}} = q(t)$$ 
и окольному
$$q_{\text{ок}} = q_{\text{пр}} + \alpha t (t_1 - t)$$

Действие Гамильтона:

$$ W = \int^{t_1}_{t_0} L(q, \dot q, t) dt$$

где $t_1 = \sqrt{10} / \omega$ и $t_0 = 0$

In [4]:
W_st = symbols('W_пр', real=True)
q_st = ans
L = (q_st.diff(t)**2 - omega**2 * q_st**2) / 2

t_end = symbols('t_end', real=True, positive=True)

W_straight = integrate(L, (t, 0, sqrt(10)/omega))

W_straight_eq = Eq(W_st, simplify(W_straight))
W_straight_eq

Eq(W_пр, -omega*q_0**2*sin(2*sqrt(10))/4)

In [5]:
W_nst, a = symbols('W_ок alpha', real=True)
q_nst = q_st + a*t*(sqrt(10)/omega - t)
L_1 = (q_nst.diff(t)**2 - omega**2 * q_nst**2) / 2

W_nstraight = integrate(L_1, (t, 0, sqrt(10)/omega))

eq_3 = Eq(W_nst, simplify(W_nstraight))
eq_3

Eq(W_ок, -omega*q_0**2*sin(2*sqrt(10))/4)

Получили, что

$$ \forall \alpha \; W_{\text{пр}} = W_{\text{ок}}$$

In [6]:
def curve(q_0=0.5, omega=np.pi/3, alpha=0.3):
    """Прямой и окольный путь"""
    t_0 = 0
    p_0 = np.array([t_0, q_0])
    
    t_1 = np.sqrt(10) / omega
    q_1 = q_0 * np.cos(omega * t_1)
    p_1 = np.array([t_1, q_1])
    
    plt.figure(figsize=(8, 6))
    
    t = np.linspace(t_0, t_1, 1000)
    
    q = q_0 * np.cos(omega * t)
    q_nst = q + alpha * t * (t_1 - t) 
    
    plt.plot(t, q, 'r-', linewidth=1.5, label=r'$\text{Прямой путь} \; q^{*}(t) = q_0 \cos \omega t$')
    plt.plot(t, q_nst, 'b--', linewidth=1.5, label=r'$\text{Окольный путь} \; q(t) = q^{*}(t) + \alpha t (t_1 - t)$')
    plt.scatter(t_0, q_0, color='black', zorder=5)
    plt.scatter(t_1, q_1, color='black', zorder=5)
    plt.title('Расширенная координатная плоскость')
    plt.xlabel('t')
    plt.ylabel(r'$\mathbf{q}$')
    plt.grid(True)
    plt.legend()
    plt.tight_layout()
    plt.show()

interact(
    curve,
    q_0=FloatSlider(min=0.1, max=2.0, step=0.1, value=0.5, description='q₀:'),
    omega=FloatSlider(min=0.1, max=2.0, step=0.1, value=np.pi/3, description='ω:'),
    alpha=FloatSlider(min=-1.0, max=1.0, step=0.1, value=0.3, description='α:')
)

interactive(children=(FloatSlider(value=0.5, description='q₀:', max=2.0, min=0.1), FloatSlider(value=1.0471975…

<function __main__.curve(q_0=0.5, omega=1.0471975511965976, alpha=0.3)>

Теперь рассмотрим зависимость действия Гамильтона от конечного момента времени, то есть интеграл с переменным верхним пределом $t_1$:

$$ W = \int^{t_1}_{t_0} L(q, \dot q, t) dt$$

Тогда действия примет вид:

In [7]:
W_r1 = simplify(integrate(L, (t, 0, t_end)))
W_r1

-omega*q_0**2*sin(2*omega*t_end)/4

In [8]:
q_nst_1 = q_st + a*t*(t_end - t)
L_1_1 = (q_nst_1.diff(t)**2 - omega**2 * q_nst_1**2) / 2
W_r2 = simplify(integrate(L_1_1, (t, 0, t_end)))
W_r2

-alpha**2*omega**2*t_end**5/60 + alpha**2*t_end**3/6 - omega*q_0**2*sin(2*omega*t_end)/4

In [9]:
def W_1(t_end, alpha=0.3, omega=np.pi/3, q_0=0.5):
    """Действие по прямому пути"""
    t, a, w, q0 = t_end, alpha, omega, q_0

    return (
        - omega * q_0**2 * np.sin(2*omega*t_end) / 4
    )

def W_2(t_end, alpha=0.3, omega=np.pi/3, q_0=0.5):
    """Действие по окольному пути"""
    a, w, q0 = alpha, omega, q_0
    t = t_end
    
    return (
        - (a**2 * w**2 * t**5) / 60
        + (a**2 * t**3) / 6
        - (w * q0**2 * np.sin(2 * w * t)) / 4
    )

In [10]:
def action(q_0=0.5, omega=np.pi/3, alpha=0.3):
    """Зависимость действия Гамильтона от t_1"""
    t_0 = 0
    p_0 = np.array([t_0, q_0])
    
    t_1 = np.sqrt(10) / omega
    q_1 = q_0 * np.cos(omega * t_1)
    p_1 = np.array([t_1, q_1])
    
    plt.figure(figsize=(8, 6))
    
    t = np.linspace(t_0, 4, 1000)
    
    q = q_0 * np.cos(omega * t)
    q_nst = q + alpha * t * (t_1 - t) 
    
    plt.plot(t, W_1(t, alpha, omega, q_0), 'r-', linewidth=1.5, label=r'$\text{Действие по} \; q^{*}(t) = q_0 \cos \omega t$')
    plt.plot(t, W_2(t, alpha, omega, q_0), 'b--', linewidth=1.5, label=r'$\text{Действие по} \; q(t) = q^{*}(t) + \alpha t (t_1 - t)$')
    plt.axvline(x=np.sqrt(10) / omega, color='black', label=r'$\text{Фокус Якоби} \; t_1 = \sqrt{10} / \omega$')
    plt.xlabel(r'$t_1$')
    plt.ylabel('W')
    plt.grid(True)
    plt.legend()
    plt.tight_layout()
    plt.show()

interact(
    action,
    q_0=FloatSlider(min=0.1, max=2.0, step=0.1, value=0.5, description='q₀:'),
    omega=FloatSlider(min=0.1, max=2.0, step=0.1, value=np.pi/3, description='ω:'),
    alpha=FloatSlider(min=-1.0, max=1.0, step=0.1, value=0.3, description='α:')
)

interactive(children=(FloatSlider(value=0.5, description='q₀:', max=2.0, min=0.1), FloatSlider(value=1.0471975…

<function __main__.action(q_0=0.5, omega=1.0471975511965976, alpha=0.3)>

Получили ожидаемый результат:

- $t_1 = \sqrt{10} / \omega$ - фокус Якоби

- для $t_1 < \sqrt{10} / \omega$ действие по прямому пути (экстремаль) меньше, чем по окольному

- для $t_1 > \sqrt{10} / \omega$ действие по прямому пути (экстремаль) больше, чем по окольному